# RGB2Point — Bangladeshi Object Point-Cloud Dataset Builder

This Colab notebook uses the released **RGB2Point** checkpoint to turn downloaded RGB images into 1,024-point XYZ clouds.

## Workflow
1. Install dependencies, clone the authors' repository, and download the released checkpoint.
2. **First-run smoke test:** upload one image, generate one point cloud, and inspect it in 3D. These two clearly marked test cells can be deleted after you are satisfied.
3. Enter an object name such as `Wooden Bench`.
4. Search DuckDuckGo Images (Bangladesh-biased region), download up to 100 distinct images, run RGB2Point, and package the images, `.ply` files, `.npy` arrays, and metadata into one ZIP.

## Important research limitations — read before using the output as training data

- The authors describe the released checkpoint as trained on **ShapeNet chairs, airplanes, and cars**. The training script's active class loop also lists those three categories. A `bench` entry in a label map is not evidence that the released checkpoint was trained on benches. Results for `Wooden Bench` and other unseen Bangladeshi objects are therefore **out-of-domain predictions**, not ground-truth reconstructions.
- RGB2Point predicts **XYZ coordinates only** (1,024 points). It does not predict per-point RGB/color, object scale in meters, material, or reliable hidden geometry.
- PointLLM is designed around **colored point clouds** and point-text instruction data. These XYZ-only files are a useful starting point, but they are not a complete PointLLM training dataset.
- MetaCLIP is an **image–text** model. Do not feed `.ply` files directly to it. Keep the downloaded RGB images and create verified captions / Bangla-English text pairs for a separate MetaCLIP fine-tuning pipeline.
- DuckDuckGo image results do not guarantee permission to reuse images. The notebook records source-page and image URLs, but does **not** verify licenses. Check each image's license/permission before training or redistributing the dataset.
- Search engines may return fewer than 100 usable, non-duplicate images, or throttle automated requests. The notebook reports the actual count and never fabricates missing samples.

**Recommended practice:** inspect the smoke-test cloud first; then manually review generated clouds and remove implausible examples before training. For high-quality local-object geometry, photos captured from multiple views or real scans with ground-truth point clouds are substantially safer than treating this single-view model's guesses as truth.

**Sources:** [RGB2Point repository](https://github.com/JaeLee18/RGB2point) · [RGB2Point WACV 2025 paper](https://arxiv.org/abs/2407.14979) · [PointLLM repository](https://github.com/InternRobotics/PointLLM) · [MetaCLIP repository](https://github.com/facebookresearch/MetaCLIP)

## 1. Install dependencies and prepare RGB2Point

In Colab, select **Runtime → Change runtime type → T4 GPU** (or another available GPU) before running these cells. The code will fall back to CPU, but generating 100 clouds will be much slower.

In [ ]:
%pip -q install "timm==1.0.15" "gdown>=5.2.0" "ddgs>=9.0.0" "plotly>=5.18.0" pandas tqdm requests pillow

In [ ]:
import os
import re
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/JaeLee18/RGB2point.git"
REPO_DIR = Path("/content/RGB2point")
CHECKPOINT_ID = "1Z5luy_833YV6NGiKjGhfsfEUyaQkgua1"
CHECKPOINT_PATH = Path("/content/pc1024_three.pth")

# Clone the official implementation.
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

# The checkpoint contains the complete model state_dict, including the ViT weights.
# Disable timm's separate ImageNet-weight download during model construction; then
# load the checkpoint's own ViT weights below.
model_file = REPO_DIR / "model.py"
model_source = model_file.read_text()
model_source, replacements = re.subn(
    r'("vit_base_patch16_224"\s*,\s*pretrained\s*=\s*)True',
    r'\1False',
    model_source,
    count=1,
)
if replacements != 1:
    raise RuntimeError("Could not safely patch timm pretrained=True in model.py; inspect the upstream repository version.")
model_file.write_text(model_source)

# Download the exact checkpoint linked in the authors' README.
if not CHECKPOINT_PATH.exists() or CHECKPOINT_PATH.stat().st_size < 1_000_000:
    import gdown
    result = gdown.download(id=CHECKPOINT_ID, output=str(CHECKPOINT_PATH), quiet=False)
    if not result or not CHECKPOINT_PATH.exists() or CHECKPOINT_PATH.stat().st_size < 1_000_000:
        raise RuntimeError(
            "Checkpoint download failed or returned an unexpectedly small file. "
            "Open the RGB2Point README and verify that the Google Drive checkpoint is still accessible."
        )

print("Repository:", REPO_DIR)
print("Checkpoint:", CHECKPOINT_PATH, f"({CHECKPOINT_PATH.stat().st_size / (1024**2):.1f} MiB)")

## 2. Load the released checkpoint

In [ ]:
import torch
import numpy as np
from PIL import Image, ImageOps
from IPython.display import display, Image as IPyImage

sys.path.insert(0, str(REPO_DIR))
from model import PointCloudNet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected. In Colab, enable a GPU runtime for practical batch processing.")

# Match the authors' inference.py architecture exactly.
model = PointCloudNet(
    num_views=1,
    point_cloud_size=1024,
    num_heads=4,
    dim_feedforward=2048,
)

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
except TypeError:  # Compatibility with older PyTorch releases.
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
# Be tolerant of a DataParallel prefix, while still loading strictly afterwards.
if isinstance(state_dict, dict) and state_dict and all(k.startswith("module.") for k in state_dict):
    state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

model.load_state_dict(state_dict, strict=True)
model.to(DEVICE)
model.eval()
print("RGB2Point checkpoint loaded successfully.")

## 3. Shared inference, export, and visualization helpers

These helpers are used by both the one-image smoke test and the 100-image dataset pipeline.

In [ ]:
import numpy as np
import torch
import plotly.graph_objects as go

_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)


def preprocess_rgb_image(image_path):
    """Replicate the repository's Resize(224), ToTensor, ImageNet normalization."""
    with Image.open(image_path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im = im.resize((224, 224), resample=Image.Resampling.BILINEAR)
        arr = np.asarray(im, dtype=np.float32) / 255.0
    arr = (arr.transpose(2, 0, 1) - _MEAN) / _STD
    tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)  # [1, 1, 3, 224, 224]
    return tensor.to(DEVICE, dtype=torch.float32)


@torch.inference_mode()
def predict_points(image_path):
    """Return the model's 1,024 predicted XYZ coordinates as float32 NumPy."""
    model_input = preprocess_rgb_image(image_path)
    prediction = model(model_input)
    points = prediction[0].detach().cpu().numpy().astype(np.float32)
    if points.shape != (1024, 3):
        raise ValueError(f"Unexpected model output shape {points.shape}; expected (1024, 3).")
    if not np.isfinite(points).all():
        raise ValueError("Model output contains NaN or infinite coordinates.")
    return points


def save_ply_xyz(points, output_path):
    """Write a standards-compatible ASCII PLY containing XYZ vertices only."""
    points = np.asarray(points, dtype=np.float32)
    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError("points must have shape (N, 3)")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        f.write("ply\nformat ascii 1.0\n")
        f.write(f"element vertex {len(points)}\n")
        f.write("property float x\nproperty float y\nproperty float z\n")
        f.write("end_header\n")
        np.savetxt(f, points, fmt="%.7f %.7f %.7f")
    return output_path


def show_point_cloud(points, title="RGB2Point prediction"):
    points = np.asarray(points)
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        mode="markers", marker=dict(size=2.2, opacity=0.85),
    )])
    fig.update_layout(
        title=title,
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
        margin=dict(l=0, r=0, t=45, b=0),
        height=620,
    )
    fig.show()


def point_cloud_stats(points):
    points = np.asarray(points)
    lo, hi = points.min(axis=0), points.max(axis=0)
    extent = hi - lo
    return {
        "points": int(len(points)),
        "x_extent": float(extent[0]),
        "y_extent": float(extent[1]),
        "z_extent": float(extent[2]),
        "min_coordinate": float(points.min()),
        "max_coordinate": float(points.max()),
    }

print("Inference helpers are ready.")

# FIRST-RUN ONLY: one-image smoke test

**Run these two test cells once before building a dataset.** Upload a representative object image (ideally one of the local Bangladeshi objects you care about), then inspect the point cloud from multiple angles. This does not calculate an accuracy score because a single internet image has no ground-truth 3D scan. It only lets you check whether the prediction looks structurally plausible.

After you are done evaluating, you can delete **the next two code cells**. Keep the setup and helper cells above, and keep the bulk pipeline cells below.

In [ ]:
# TEST CELL 1/2 — upload exactly one RGB image
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No image was uploaded.")

smoke_test_filename = next(iter(uploaded))
SMOKE_TEST_IMAGE = f"/content/{smoke_test_filename}"
print("Smoke-test image:", SMOKE_TEST_IMAGE)
display(IPyImage(filename=SMOKE_TEST_IMAGE))

smoke_test_points = predict_points(SMOKE_TEST_IMAGE)
SMOKE_TEST_PLY = "/content/rgb2point_smoke_test.ply"
save_ply_xyz(smoke_test_points, SMOKE_TEST_PLY)
print("Point-cloud statistics:", point_cloud_stats(smoke_test_points))
print("Saved:", SMOKE_TEST_PLY)

In [ ]:
# TEST CELL 2/2 — inspect the generated point cloud interactively
show_point_cloud(smoke_test_points, title=f"Smoke test: {smoke_test_filename}")

# Optional: download this single PLY for inspection in CloudCompare / MeshLab.
# from google.colab import files
# files.download(SMOKE_TEST_PLY)

## 4. Dataset settings — change only the object name for a new class

`DDG_REGION="bd-en"` biases DuckDuckGo toward Bangladesh. With `ADD_BANGLADESH_TERM=True`, the first search query also appends the word Bangladesh. These settings improve locality but cannot guarantee that every result depicts a Bangladeshi-made or locally distinctive object.

In [ ]:
# EDIT THIS for each dataset you want to generate.
OBJECT_QUERY = "Wooden Bench"

TARGET_IMAGES = 100
DDG_REGION = "bd-en"              # Bangladesh-focused DuckDuckGo region
ADD_BANGLADESH_TERM = True         # First query becomes e.g. "Wooden Bench Bangladesh"
MIN_IMAGE_SIDE = 128               # Reject tiny thumbnails/icons
MAX_IMAGE_MB = 15                  # Reject oversized downloads
REQUEST_TIMEOUT = 25

OUTPUT_ROOT = Path("/content/rgb2point_datasets")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

assert OBJECT_QUERY.strip(), "Please set OBJECT_QUERY to a non-empty object name."
print(f"Will search for up to {TARGET_IMAGES} images for: {OBJECT_QUERY!r}")
print("Output root:", OUTPUT_ROOT)

## 5. Search DuckDuckGo Images and download distinct, valid images

The collector tries the Bangladesh-biased query first, then a couple of fallbacks. It deduplicates exact URLs, file hashes, and near-identical perceptual hashes. It stops after 100 successful downloads or after exhausting the search results.

In [ ]:
import hashlib
import io
import time
import pandas as pd
import requests
from datetime import datetime
from ddgs import DDGS
from tqdm.auto import tqdm
from PIL import Image, ImageOps


def slugify(text):
    slug = re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")
    return slug or "object"


def ddg_image_results(query, max_results=100):
    """Return DuckDuckGo image results; errors are handled by caller."""
    client = DDGS()
    return list(client.images(
        query,
        region=DDG_REGION,
        safesearch="moderate",
        max_results=max_results,
    ))


def dhash_64(pil_image):
    small = np.asarray(pil_image.convert("L").resize((9, 8), Image.Resampling.LANCZOS))
    bits = small[:, 1:] > small[:, :-1]
    value = 0
    for bit in bits.flatten():
        value = (value << 1) | int(bit)
    return value


def hamming_distance(a, b):
    return (int(a) ^ int(b)).bit_count()


def fetch_and_save_image(candidate, output_path):
    image_url = candidate.get("image") or candidate.get("image_url") or candidate.get("thumbnail")
    if not image_url:
        raise ValueError("Search result has no usable image URL")
    headers = {"User-Agent": "Mozilla/5.0 (compatible; RGB2PointDatasetResearch/1.0)"}
    response = requests.get(image_url, headers=headers, timeout=REQUEST_TIMEOUT, stream=True)
    response.raise_for_status()
    content = response.content
    if len(content) > MAX_IMAGE_MB * 1024 * 1024:
        raise ValueError(f"Image exceeds {MAX_IMAGE_MB} MB")
    if len(content) < 2_000:
        raise ValueError("Image response is too small")

    # Verify the image first, then reopen and normalize it to RGB JPEG.
    with Image.open(io.BytesIO(content)) as im_check:
        im_check.verify()
    with Image.open(io.BytesIO(content)) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        width, height = im.size
        if min(width, height) < MIN_IMAGE_SIDE:
            raise ValueError(f"Image too small ({width}x{height})")
        phash = dhash_64(im)
        im.save(output_path, format="JPEG", quality=94, optimize=True)
    return {
        "image_url": image_url,
        "source_page_url": candidate.get("url", ""),
        "title": candidate.get("title", ""),
        "source": candidate.get("source", ""),
        "original_width": int(width),
        "original_height": int(height),
        "sha256": hashlib.sha256(content).hexdigest(),
        "dhash": phash,
    }


DATASET_STAMP = datetime.utcnow().strftime("%Y%m%d_%H%M%S_utc")
DATASET_DIR = OUTPUT_ROOT / f"{slugify(OBJECT_QUERY)}_{DATASET_STAMP}"
IMAGES_DIR = DATASET_DIR / "images"
POINTS_DIR = DATASET_DIR / "pointclouds"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
POINTS_DIR.mkdir(parents=True, exist_ok=True)

search_queries = []
if ADD_BANGLADESH_TERM:
    search_queries.append(f"{OBJECT_QUERY.strip()} Bangladesh")
search_queries.extend([OBJECT_QUERY.strip(), f"{OBJECT_QUERY.strip()} object photo"])
search_queries = list(dict.fromkeys(q for q in search_queries if q))

all_candidates = []
seen_candidate_urls = set()
for search_query in search_queries:
    print(f"Searching DuckDuckGo Images: {search_query!r}")
    try:
        results = ddg_image_results(search_query, max_results=100)
        print("  results returned:", len(results))
        for result in results:
            url = result.get("image") or result.get("image_url") or result.get("thumbnail")
            if url and url not in seen_candidate_urls:
                seen_candidate_urls.add(url)
                result = dict(result)
                result["query_used"] = search_query
                all_candidates.append(result)
    except Exception as exc:
        print("  Search request failed:", repr(exc))
    if len(all_candidates) >= TARGET_IMAGES * 2:
        break

print("Unique image candidates collected:", len(all_candidates))

accepted = []
seen_sha256 = set()
seen_hashes = []
failures = []
for candidate in tqdm(all_candidates, desc="Downloading and validating images"):
    if len(accepted) >= TARGET_IMAGES:
        break
    image_number = len(accepted) + 1
    image_path = IMAGES_DIR / f"{slugify(OBJECT_QUERY)}_{image_number:04d}.jpg"
    try:
        details = fetch_and_save_image(candidate, image_path)
        if details["sha256"] in seen_sha256:
            image_path.unlink(missing_ok=True)
            continue
        # Remove near-identical copies while retaining visually different examples.
        if any(hamming_distance(details["dhash"], old_hash) <= 2 for old_hash in seen_hashes):
            image_path.unlink(missing_ok=True)
            continue
        seen_sha256.add(details["sha256"])
        seen_hashes.append(details["dhash"])
        accepted.append({
            "sample_id": f"{slugify(OBJECT_QUERY)}_{len(accepted)+1:04d}",
            "object_label": OBJECT_QUERY,
            "image_path": str(image_path),
            "search_query": candidate.get("query_used", ""),
            **{k: v for k, v in details.items() if k != "dhash"},
            "license_status": "NOT_VERIFIED_CHECK_BEFORE_TRAINING",
            "manual_review_status": "PENDING",
            "manual_review_notes": "",
        })
    except Exception as exc:
        failures.append({"image_url": candidate.get("image", ""), "error": str(exc)})
        image_path.unlink(missing_ok=True)
    time.sleep(0.12)

print(f"\nSuccessfully downloaded: {len(accepted)} / {TARGET_IMAGES}")
print("Rejected/failed candidates:", len(failures))
if len(accepted) < TARGET_IMAGES:
    print("WARNING: DuckDuckGo did not provide 100 usable distinct images in this run. "
          "The next cells will process the images actually downloaded; no fake samples are added.")
if not accepted:
    raise RuntimeError("No usable images were downloaded. Try another query or rerun later if DDG rate-limited the search.")

pd.DataFrame(accepted).drop(columns=["image_path"], errors="ignore").head()

## 6. Run RGB2Point over every downloaded image

In [ ]:
from tqdm.auto import tqdm

successful_records = []
for record in tqdm(accepted, desc="Generating point clouds"):
    sample_id = record["sample_id"]
    image_path = record["image_path"]
    ply_path = POINTS_DIR / f"{sample_id}.ply"
    npy_path = POINTS_DIR / f"{sample_id}.npy"
    try:
        points = predict_points(image_path)
        save_ply_xyz(points, ply_path)
        np.save(npy_path, points)
        stats = point_cloud_stats(points)
        record.update({
            "generation_status": "success",
            "point_count": stats["points"],
            "pointcloud_ply": str(ply_path.relative_to(DATASET_DIR)),
            "pointcloud_npy": str(npy_path.relative_to(DATASET_DIR)),
            "x_extent": stats["x_extent"],
            "y_extent": stats["y_extent"],
            "z_extent": stats["z_extent"],
            "min_coordinate": stats["min_coordinate"],
            "max_coordinate": stats["max_coordinate"],
            "generation_error": "",
        })
    except Exception as exc:
        record.update({
            "generation_status": "failed",
            "point_count": 0,
            "pointcloud_ply": "",
            "pointcloud_npy": "",
            "generation_error": repr(exc),
        })
    successful_records.append(record)

manifest = pd.DataFrame(successful_records)
manifest_path = DATASET_DIR / "manifest.csv"
manifest.drop(columns=["image_path", "sha256"], errors="ignore").to_csv(manifest_path, index=False)

success_count = int((manifest["generation_status"] == "success").sum())
print(f"Point clouds generated: {success_count} / {len(accepted)}")
display(manifest[["sample_id", "object_label", "generation_status", "point_count", "source_page_url"]].head(10))

## 7. Write dataset notes and create the ZIP archive

In [ ]:
import shutil

readme_text = f"""RGB2Point dataset export
=======================
Object query: {OBJECT_QUERY}
Images downloaded: {len(accepted)}
Point clouds generated successfully: {success_count}
Model: RGB2Point, released checkpoint pc1024_three.pth
Checkpoint source: https://github.com/JaeLee18/RGB2point
Checkpoint training domain stated by authors: ShapeNet chairs, airplanes, and cars
Output: 1,024 XYZ points per successful sample; coordinates are model-space, not meters.
Point colors: not predicted by this RGB2Point model.

Research cautions
-----------------
1. Results for unseen classes are out-of-domain predictions, not ground-truth scans.
2. Manually inspect each point cloud before using it for training.
3. Image reuse licenses were not verified. Use source_page_url and image_url in manifest.csv
   to check permissions and preserve attribution as required.
4. PointLLM commonly uses colored point clouds and point-text instruction pairs. These
   exports contain XYZ only and need additional color/annotation preparation.
5. MetaCLIP is an image-text model. Use the paired RGB images with reviewed captions for
   its training data; do not feed PLY files to MetaCLIP directly.

Folder layout
-------------
images/       Downloaded source images (normalized JPEG copies)
pointclouds/  One .ply and one .npy per successful inference
manifest.csv  Per-sample URLs, source text, dimensions, result paths, and status
README.txt    This file
"""
(DATASET_DIR / "README.txt").write_text(readme_text, encoding="utf-8")

zip_path = Path(shutil.make_archive(
    str(DATASET_DIR),
    "zip",
    root_dir=str(OUTPUT_ROOT),
    base_dir=DATASET_DIR.name,
))
print("Dataset folder:", DATASET_DIR)
print("ZIP archive:", zip_path)
print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MiB")

In [ ]:
# Download the packaged dataset to your computer.
from google.colab import files
files.download(str(zip_path))

## What you get — and what still needs to happen for fine-tuning

The ZIP is a **paired RGB-image / generated-XYZ-cloud collection**, not a finished training set for both models.

- **For PointLLM:** review geometry, add/derive color information if the selected PointLLM pipeline requires colored points, and create verified point-text instruction/answer annotations. The original RGB image can help with manual annotation, but this notebook does not infer reliable point-wise colors.
- **For MetaCLIP / MetaCLIP 2:** build image-caption pairs from the original RGB images (including Bangla and English descriptions where appropriate), verify the label and locality, and fine-tune the image-text model using its supported image-text training path. The generated point clouds are not direct MetaCLIP inputs.
- **For both:** split by source/object instance where possible to reduce near-duplicate leakage; record licensing; manually reject incorrect search results and poor reconstructions; and maintain a held-out validation set.

A practical next step after smoke-testing this checkpoint is to evaluate it on a small, manually selected set of local-object images. If the shapes are not plausible, do not use the generated clouds as pseudo-ground-truth—switch to a reconstruction model trained on broader object categories or capture multi-view scans.